In [ ]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestClassifier
X_full=pd.read_csv('train.csv',index_col='id')
X_test_full=pd.read_csv('test.csv',index_col='id')
X_full.dropna(axis=0,subset=['quality'],inplace=True)
y=X_full.quality
X_full.drop(['quality'],axis=1,inplace=True)
print(X_full)
X_train_full,X_valid_full,y_train,y_valid=train_test_split(X_full,y,train_size=0.8,test_size=0.2,random_state=31)
categorical_cols=[cname for cname in X_train_full.columns if X_train_full[cname].nunique()<10 and X_train_full[cname].dtype=="object"]
numerical_cols=[cname for cname in X_train_full.columns if X_train_full[cname].dtype in ['int64','float64']]
my_cols= categorical_cols+numerical_cols
X_train=X_train_full[my_cols].copy()
X_valid=X_valid_full[my_cols].copy()
X_test=X_test_full[my_cols].copy()

      fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
id                                                                              
0               7.4             0.700         0.00             1.9      0.076   
1               7.8             0.880         0.00             2.6      0.098   
2               7.8             0.760         0.04             2.3      0.092   
4               7.4             0.700         0.00             1.9      0.076   
5               7.4             0.660         0.00             1.8      0.075   
...             ...               ...          ...             ...        ...   
1592            6.3             0.510         0.13             2.3      0.076   
1593            6.8             0.620         0.08             1.9      0.068   
1594            6.2             0.600         0.08             2.0      0.090   
1595            5.9             0.550         0.10             2.2      0.062   
1597            5.9         

In [65]:
numerical_transformer=SimpleImputer(strategy="mean")
categorical_transformer=Pipeline(
    steps=[
        ("onehot",OneHotEncoder(handle_unknown='ignore')),
        ("imputer",SimpleImputer(strategy="most_frequent"))
    ]
)
preprocessor=ColumnTransformer(
    transformers=[
        ('num',numerical_transformer,numerical_cols),
        ('cat',categorical_transformer,categorical_cols)
    ]
)
model=RandomForestRegressor()

In [69]:
param_grid=[
    {
        'model__random_state':[31],
        'model__n_estimators':[95,96,97,98,99],
        'model__max_depth':[12,13,14],
        'model__max_leaf_nodes':[33,34,35,36,37]
    }
]
#{'model__learning_rate': 0.09, 'model__max_depth': 5, 'model__n_estimators': 50, 'model__random_state': 0}
#-0.6579950898885727
#RMSE: 0.5902547836303711

#{'model__learning_rate': 0.11, 'model__max_depth': 3, 'model__n_estimators': 25, 'model__random_state': 0}
#-0.6636741757392883
#RMSE: 0.5167497396469116

#{'model__learning_rate': 0.08, 'model__max_depth': 3, 'model__n_estimators': 40, 'model__random_state': 0}
#-0.6487062096595764





#{'model__max_depth': 5, 'model__max_leaf_nodes': 50, 'model__n_estimators': 46, 'model__random_state': 0}
#-0.6588484016971903

#{'model__max_depth': 9, 'model__max_leaf_nodes': 25, 'model__n_estimators': 75, 'model__random_state': 0}
#-0.6548712190461211

#{'model__max_depth': 13, 'model__max_leaf_nodes': 36, 'model__n_estimators': 99, 'model__random_state': 0}
#-0.6535724315022434
my_pipeline=Pipeline(
    steps=[
        ('preprocessor',preprocessor),
        ('model',model)
    ]
)
grid=GridSearchCV(
    estimator=my_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)
grid.fit(X_full,y)

print(grid.best_params_)
print(grid.best_score_)
best_model=grid.best_estimator_

{'model__max_depth': 13, 'model__max_leaf_nodes': 34, 'model__n_estimators': 97, 'model__random_state': 31}
-0.6565593011299542


In [70]:
finalmodel=RandomForestRegressor(
    random_state=31,
    max_leaf_nodes=34,
    max_depth=13,
    n_estimators=97
)
finalpipe=Pipeline(
    steps=[
        ('preprocessor',preprocessor),
        ('model',finalmodel)
    ]
)
finalpipe.fit(X_full,y)
preds_test=finalpipe.predict(X_test_full)

In [71]:
output=pd.DataFrame({'id':X_test.index,'quality':preds_test})
output.to_csv('submission.csv',index=False)